In [9]:
import pandas as pd
import holidays

cpi = pd.read_csv('cpi_daily.csv')
cpi['Date'] = pd.to_datetime(cpi['Date'], format='%d/%m/%y', errors='coerce')
cpi = cpi.dropna(subset=['Date', 'CPI'])

dxy = pd.read_csv('DXY.csv')[['Date', 'Price']]
dxy = dxy.rename(columns={'Price': 'DXY'})
dxy['Date'] = pd.to_datetime(dxy['Date'], errors='coerce')
dxy = dxy.dropna(subset=['Date', 'DXY'])

ls = pd.read_csv('lslnh.csv')[['date', 'rate_overnight_percent_per_annum']]
ls = ls.rename(columns={'date': 'Date', 'rate_overnight_percent_per_annum': 'Lãi suất liên ngân hàng'})
ls['Date'] = pd.to_datetime(ls['Date'], errors='coerce')
ls = ls.dropna(subset=['Date', 'Lãi suất liên ngân hàng'])

pe = pd.read_csv('pe_ratio_daily.csv')
pe = pe.rename(columns={'P/E ratio': 'P/E'})
pe['Date'] = pd.to_datetime(pe['Date'], format='%d/%m/%y', errors='coerce')
pe = pe.dropna(subset=['Date', 'P/E'])

us10y = pd.read_csv('US10Y.csv')[['Date', 'Price']]
us10y = us10y.rename(columns={'Price': 'US10Y'})
us10y['Date'] = pd.to_datetime(us10y['Date'], errors='coerce')
us10y = us10y.dropna(subset=['Date', 'US10Y'])

usd_vnd = pd.read_csv('USD-VND.csv')[['Date', 'Price']]
usd_vnd = usd_vnd.rename(columns={'Price': 'USD-VND'})
usd_vnd['Date'] = pd.to_datetime(usd_vnd['Date'], errors='coerce')
usd_vnd['USD-VND'] = usd_vnd['USD-VND'].astype(str).str.replace(',', '').astype(float)
usd_vnd = usd_vnd.dropna(subset=['Date', 'USD-VND'])

vn10y = pd.read_csv('VN10Y.csv')[['Date', 'Price']]
vn10y = vn10y.rename(columns={'Price': 'VN10Y'})
vn10y['Date'] = pd.to_datetime(vn10y['Date'], errors='coerce')
vn10y = vn10y.dropna(subset=['Date', 'VN10Y'])

df_spread = pd.merge(us10y, vn10y, on='Date', how='inner')
df_spread['US10Y - VN10Y'] = df_spread['US10Y'] - df_spread['VN10Y']
df_spread = df_spread[['Date', 'US10Y - VN10Y']]

start_date = pd.to_datetime('2017-01-01')
end_date = pd.to_datetime('2026-6-5')

daily_range = pd.date_range(start=start_date, end=end_date, freq='D')
merged_df = pd.DataFrame({'Date': daily_range})

merged_df = merged_df[merged_df['Date'].dt.dayofweek < 5]

vn_holidays = holidays.VN(years=range(2017, 2027))
merged_df = merged_df[~merged_df['Date'].isin(vn_holidays)]

dfs = [df_spread, cpi, dxy, ls, pe, usd_vnd]
for df in dfs:
    merged_df = pd.merge(merged_df, df, on='Date', how='left')

data_cols = ['US10Y - VN10Y', 'CPI', 'DXY', 'Lãi suất liên ngân hàng', 'P/E', 'USD-VND']
merged_df[data_cols] = merged_df[data_cols].ffill().bfill()

merged_df = merged_df.sort_values('Date').reset_index(drop=True)
merged_df['Date'] = merged_df['Date'].dt.strftime('%d/%m/%y')

final_cols = ['Date', 'US10Y - VN10Y', 'CPI', 'DXY', 'Lãi suất liên ngân hàng', 'P/E', 'USD-VND']
merged_df = merged_df[final_cols]

output_filename = 'Macro_Data_2017_2026_Final.csv'
merged_df.to_csv(output_filename, index=False)

C:\Users\HP 9480m\AppData\Local\Temp\ipykernel_26416\3451825575.py:52: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  merged_df = merged_df[~merged_df['Date'].isin(vn_holidays)]
